# SciPy — numerical methods on top of NumPy

**What this library does in one sentence:** SciPy bundles the classic numerical-methods toolbox (ODE solvers, optimisers, FFTs, interpolation, linear algebra) on top of NumPy arrays.

For this project we use exactly one corner of SciPy: `scipy.integrate.solve_ivp`, the workhorse for solving differential equations. That's it. Cover this well and you're done.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp

## 1. What is an ODE? (with a non-pendulum example first)

An **ordinary differential equation** (ODE) is an equation that tells you how a quantity *changes* given its current value, not the value itself. You give the solver an *initial value* and it walks forward in time.

**Radioactive decay** is the simplest possible example:

$$\frac{dN}{dt} = -\lambda N$$

In words: "the rate of decay is proportional to how much is left." If you know `N(0) = 100`, the solver gives you `N(t)` for any future `t`.

Note what changes: the **rate** `dN/dt` is what you write down. The solver figures out `N(t)` by integrating that rate over and over.

In [ ]:
# the RHS function: returns the derivative, given (t, y)
def decay(t, N):
    lam = 0.5         # decay constant
    return -lam * N

sol = solve_ivp(decay, t_span=(0, 10), y0=[100], t_eval=np.linspace(0, 10, 100))

plt.plot(sol.t, sol.y[0])
plt.xlabel('time'); plt.ylabel('N(t)')
plt.title('radioactive decay'); plt.grid(alpha=0.3)
plt.show()

That's the whole pattern. **Write the derivative, hand it to `solve_ivp`, get back the trajectory.** Everything else is variations on this.

## 2. `solve_ivp` — full signature

```python
sol = solve_ivp(fun, t_span, y0, method='RK45', t_eval=None, max_step=np.inf, args=None)
```

| Parameter   | Meaning |
|-------------|---------|
| `fun`       | The RHS function. **Must have signature `f(t, y)` returning `dy/dt`.** `t` is a scalar; `y` is an array (shape `(n,)` for an n-dimensional state). |
| `t_span`    | Tuple `(t0, t_final)` — when to start and stop integrating. |
| `y0`        | Initial state, an array of length `n`. |
| `method`    | Integration scheme: `'RK45'` (default, good for most), `'RK23'` (cheaper, less accurate), `'DOP853'` (high accuracy for smooth problems), `'LSODA'` (stiff problems). |
| `t_eval`    | Optional array of times where you want output. If omitted, the solver picks adaptive steps and you get whatever it chose. |
| `max_step`  | Hard upper bound on the internal step size. Crucial for fast/stiff dynamics. |
| `args`      | Tuple of extra arguments to pass to `fun` — e.g. `args=(mass, spring_k)`. |

### The RHS function signature

**Always `f(t, y)`, in that order, even if you don't use `t`.** `y` is a 1-D NumPy array of length `n`; you must return an iterable of length `n` (a list or array of the same shape).

```python
def f(t, y):
    # unpack the state into named pieces — much easier to read
    x, v = y
    # write each derivative
    dxdt = v
    dvdt = -9.81           # gravity, ignoring everything else
    return [dxdt, dvdt]    # return list (or np.array) of length n
```

If you forget the `t` parameter — write `def f(y)` — you'll get a confusing TypeError. Always include it.

### The return value

`sol` is an object (a `Bunch`) with these fields you care about:

| Field           | Meaning |
|-----------------|---------|
| `sol.t`         | 1-D array of timepoints (length `len(t_eval)` if provided, otherwise solver-chosen). |
| `sol.y`         | 2-D array, **shape `(n_states, n_timepoints)`** — one row per state variable. |
| `sol.success`   | `True` if integration completed; `False` if it bailed. |
| `sol.message`   | Human-readable status — read this when something looks wrong. |
| `sol.nfev`      | How many times `fun` was called — a rough cost measure. |

### `method='RK45'` — what's RK?

**Runge-Kutta** is a family of step-by-step integrators. To advance from `t` to `t+dt`, RK4 evaluates the derivative `f(t, y)` at 4 cleverly-chosen points (start, two midpoints, end), then takes a weighted average of those slopes. That weighted average is the step.

`RK45` is the *Dormand-Prince* variant: it computes both a 4th-order and a 5th-order estimate, compares them, and uses the difference to **automatically shrink or grow the step size**. That's the "adaptive" part.

For a smooth system like ours, `RK45` defaults are excellent and you rarely change them.

### `max_step` — why it matters

Adaptive solvers can take *huge* steps when the dynamics are calm. For a pendulum near its equilibrium, this is fine. But if you whack the cart and the pole spins fast, the solver may underestimate how fast `y` is changing between checks and miss the spike.

Setting `max_step=0.01` says: "never advance more than 10 ms internally, even if you think you can." This is the single most common tweak you'll make.

## 3. `RK45` vs `RK23` vs `DOP853` — when to use which

| Method   | Order | Cost / step | Use when |
|----------|-------|-------------|----------|
| `RK23`   | 2-3   | low         | rough quick-look, low-accuracy ok. Rarely the right answer. |
| `RK45`   | 4-5   | medium      | **default — use this unless you have a reason not to.** Great for smooth mechanical systems. |
| `DOP853` | 8     | high        | you need very high accuracy and the system is smooth (e.g. orbital mechanics over long times). |
| `LSODA`  | varies | medium     | the system is *stiff* — derivatives change on wildly different time scales. Pendulums are NOT stiff. |

**For this project: `RK45` everywhere.** It's the right call for the cart-pole.

## 4. Example 1: simple harmonic oscillator

A mass on a spring: $\ddot{x} = -\frac{k}{m} x$. Analytic solution is $x(t) = x_0 \cos(\omega t)$ with $\omega = \sqrt{k/m}$ — easy to verify the solver.

In [ ]:
k, m = 4.0, 1.0
omega = np.sqrt(k / m)

def sho(t, y):
    x, v = y
    return [v, -k/m * x]

sol = solve_ivp(sho, (0, 10), y0=[1.0, 0.0], t_eval=np.linspace(0, 10, 500))

plt.plot(sol.t, sol.y[0],            label='numerical')
plt.plot(sol.t, np.cos(omega*sol.t), '--', label='analytic')
plt.xlabel('t'); plt.ylabel('x'); plt.legend(); plt.grid(alpha=0.3)
plt.show()

The two curves should sit on top of each other. If they don't, your RHS is wrong.

## 5. Example 2: simple pendulum (nonlinear)

A pendulum of length `L` swinging under gravity, no cart: $\ddot{\theta} = -\frac{g}{L} \sin\theta$.

This is nonlinear (because of `sin`). For small `θ`, `sin θ ≈ θ` and you recover SHO. For large swings, the period stretches.

In [ ]:
g, L = 9.81, 1.0

def pendulum(t, y):
    th, thd = y
    return [thd, -g/L * np.sin(th)]

for th0 in [0.1, 1.0, 2.5]:                # tiny / medium / huge swing
    sol = solve_ivp(pendulum, (0, 5), y0=[th0, 0.0],
                    t_eval=np.linspace(0, 5, 500))
    plt.plot(sol.t, sol.y[0], label=f'theta0 = {th0:.1f} rad')

plt.xlabel('t'); plt.ylabel('theta'); plt.legend(); plt.grid(alpha=0.3)
plt.show()

Notice how the larger swings have a longer period — that's the nonlinearity. A linear analysis would miss it.

## 6. Example 3: cart-pole (this project)

Four-state system: cart position `x`, cart velocity `ẋ`, pole angle `θ`, pole rate `θ̇`. The EOM is derived in the [sympy tutorial](../03_sympy/sympy_tutorial.ipynb); here we just use the numerical form.

In [ ]:
M, m, L, g = 1.0, 0.3, 0.8, 9.81

def cartpole(t, y, F=0.0):
    x, xd, th, thd = y
    s, c = np.sin(th), np.cos(th)
    denom = M + m * s**2
    x_dd  = (F - m*g*s*c + m*L*thd**2 * s) / denom
    th_dd = ((M+m)*g*s - m*L*thd**2*s*c - F*c) / (L * denom)
    return [xd, x_dd, thd, th_dd]

sol = solve_ivp(cartpole,
                t_span   = (0, 5),
                y0       = [0.0, 0.0, np.radians(5), 0.0],   # 5 deg lean
                method   = 'RK45',
                t_eval   = np.linspace(0, 5, 501),
                max_step = 0.01,                              # important — fast pole dynamics
                args     = (0.0,))                            # F = 0 → passive simulation

print('success?  ', sol.success)
print('message:  ', sol.message)
print('sol.y.shape =', sol.y.shape)        # (4, 501) — 4 states × 501 timepoints

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(8, 4), sharex=True)
axes[0].plot(sol.t, sol.y[0]);                  axes[0].set_ylabel('cart x (m)')
axes[1].plot(sol.t, np.degrees(sol.y[2]));      axes[1].set_ylabel('theta (deg)')
axes[1].set_xlabel('t (s)')
for ax in axes: ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

## 7. Reading `sol.y` — shape `(n_states, n_timepoints)`

The single most confusing thing about `solve_ivp` for newcomers is that `sol.y` is **transposed** relative to how you'd preallocate ("rows = time"). SciPy's convention is **rows = state variables**.

For the cart-pole, `sol.y.shape == (4, 501)`:

```python
x_array     = sol.y[0]      # cart position over time
xd_array    = sol.y[1]      # cart velocity
theta_array = sol.y[2]      # pole angle
thetad_array= sol.y[3]      # pole rate
```

To get the state at the i-th timepoint, use `sol.y[:, i]` (all rows, column i).

If you'd prefer the (time, state) layout (matches typical preallocation), just transpose: `states = sol.y.T` gives shape `(501, 4)`.

## Functions used in this project — quick reference

| Function | Signature | What it does |
|----------|-----------|--------------|
| `solve_ivp` | `solve_ivp(fun, t_span, y0, method='RK45', t_eval=None, max_step=inf, args=None)` | Integrate `dy/dt = fun(t, y)` from `t_span[0]` to `t_span[1]` starting at `y0`. |

**RHS contract:** `fun(t, y)` → returns array of length `len(y)`.

**Result fields:**

| Field | Shape | Meaning |
|-------|-------|---------|
| `sol.t`       | `(n_t,)`        | times where state was recorded |
| `sol.y`       | `(n_states, n_t)` | trajectory; `sol.y[i]` is variable `i` over time |
| `sol.success` | `bool`          | did integration complete? |
| `sol.message` | `str`           | human-readable status |

**Method picker:**

| Method   | Use when |
|----------|----------|
| `RK45`   | default — use this. |
| `RK23`   | quick low-accuracy check. |
| `DOP853` | very high accuracy on smooth problems. |
| `LSODA`  | stiff systems (not us). |